# Part 1: Data Foundation and Descriptive Analysis
## Week 2: Data Setup and Inventory

**Course:** DATA 6700 Independent Study, Summer 2026  
**Period:** May 25-31  
**Goal:** Locate and inventory all sixteen CHR analytic files, document source notes, flag known comparability issues, and record initial research questions.

**Notebook Sections**
1. Environment Setup
2. Path Configuration
3. Per-File Quick Audit
4. Release Year vs. Measurement Year Reference Table
5. Known Comparability Flags
6. Data Source Notes
7. Initial Research Questions
8. Notebook Summary

**Reproducibility Note:** All paths are relative to `DATA_DIR` defined in Section 2. Set that variable to the folder containing the sixteen CSV files and re-run top-to-bottom.

---
## Section 1: Environment Setup

### Description
Import all libraries used in this notebook and configure display settings.

In [1]:
import os
import warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

In [3]:
print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

pandas 2.3.3  |  numpy 2.3.5


### Summary
Libraries imported and display settings configured.

---
## Section 2: Path Configuration

### Description
Define the directory structure used throughout this notebook and across all Part 1, 2, and 3 notebooks.

Raw CHR files are stored in the project GitHub repository under `data/raw/`. The cells below download each file from GitHub on first run and cache it locally so subsequent runs load from disk. No manual downloading is required.

**Repository:** https://github.com/sphillips32/A-Longitudinal-Study-of-County-Health-Insurance-Coverage-2010-2025  
**Branch:** main  
**Raw files path:** data/raw/

The consolidated panel file produced in Week 3 will be saved to `data/chr_panel.csv` and used as the primary data source for all subsequent notebooks.

In [4]:
import urllib.request

In [5]:
# GitHub repo base URL for raw file access
GITHUB_RAW_BASE = (
    'https://raw.githubusercontent.com/sphillips32/'
    'A-Longitudinal-Study-of-County-Health-Insurance-Coverage-2010-2025/'
    'main/data/raw'
)

# Local cache directory: files are downloaded here on first run
DATA_DIR   = Path('data/raw')
OUTPUT_DIR = Path('outputs')

In [6]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
RELEASE_YEARS = list(range(2010, 2026))

print(f'GitHub source : {GITHUB_RAW_BASE}')
print(f'Local cache   : {DATA_DIR.resolve()}')
print(f'Output dir    : {OUTPUT_DIR.resolve()}')
print(f'Release years : {RELEASE_YEARS[0]}-{RELEASE_YEARS[-1]}  ({len(RELEASE_YEARS)} files expected)')

GitHub source : https://raw.githubusercontent.com/sphillips32/A-Longitudinal-Study-of-County-Health-Insurance-Coverage-2010-2025/main/data/raw
Local cache   : C:\Users\sethp\OneDrive\Documents\Education\Independent Study\data\raw
Output dir    : C:\Users\sethp\OneDrive\Documents\Education\Independent Study\outputs
Release years : 2010-2025  (16 files expected)


In [8]:
# Exact filename for each release year as stored in the repo
YEAR_FILE_MAP = {
    2010: 'analytic_data2010.csv',
    2011: 'analytic_data2011.csv',
    2012: 'analytic_data2012.csv',
    2013: 'analytic_data2013.csv',
    2014: 'analytic_data2014.csv',
    2015: 'analytic_data2015.csv',
    2016: 'analytic_data2016.csv',
    2017: 'analytic_data2017.csv',
    2018: 'analytic_data2018_0.csv',
    2019: 'analytic_data2019.csv',
    2020: 'analytic_data2020_0.csv',
    2021: 'analytic_data2021.csv',
    2022: 'analytic_data2022.csv',
    2023: 'analytic_data2023_0.csv',
    2024: 'analytic_data2024.csv',
    2025: 'analytic_data2025_v3.csv',
}

In [9]:
def build_github_url(year, base=GITHUB_RAW_BASE):
    """
    Build the raw GitHub URL for a given release year using the explicit file map.
    Update YEAR_FILE_MAP above if any filenames change in the repo.
    """
    filename = YEAR_FILE_MAP[year]
    return f'{base}/{filename}'

In [10]:
def fetch_chr_file(year, data_dir=DATA_DIR):
    """
    Return the local Path for a CHR file, downloading from GitHub if not cached.
    Skips the download if the file already exists locally.
    """
    url        = build_github_url(year)
    local_path = data_dir / url.split('/')[-1]

    if local_path.exists():
        return local_path, 'cached'

    try:
        urllib.request.urlretrieve(url, local_path)
        return local_path, 'downloaded'
    except Exception as e:
        return None, f'ERROR: {e}'

In [11]:
fetch_results = []

for yr in RELEASE_YEARS:
    path, status = fetch_chr_file(yr)
    fetch_results.append({'release_year': yr, 'status': status, 'path': str(path) if path else None})
    print(f'{yr}  {status}')

2010  cached
2011  cached
2012  cached
2013  cached
2014  cached
2015  cached
2016  cached
2017  cached
2018  cached
2019  cached
2020  cached
2021  cached
2022  cached
2023  cached
2024  cached
2025  cached


In [12]:
fetch_df = pd.DataFrame(fetch_results)

errors = fetch_df[fetch_df['status'].str.startswith('ERROR')]

if not errors.empty:
    print('Files that could not be fetched:')
    print(errors[['release_year', 'status']].to_string(index=False))
    print()
    print('Check that the filename pattern in build_github_url() matches your repo.')
else:
    print(f'All {len(RELEASE_YEARS)} files ready.')

All 16 files ready.


### Summary
Files are sourced from the project GitHub repository and cached locally under `data/raw/`. On first run all 16 files are downloaded. On subsequent runs they load from the local cache. Any fetch errors indicate a filename mismatch in `build_github_url()` and should be resolved before proceeding. The consolidated panel will be written to `data/chr_panel.csv` during Week 3.

---
## Section 3: QA Each File

### Description
Read the header and a small sample from each file to record row counts, column counts, and the exact names of the FIPS and uninsured-rate columns. CHR column names changed across releases, so this audit surfaces those differences before panel construction begins in Week 3.

In [13]:
# Known FIPS column name variants across CHR releases
FIPS_VARIANTS = ['fipscode', 'fips', '5-digit fips code', 'county fips']

# Known uninsured column name variants
# v059_rawvalue = % uninsured all ages (2015+ releases)
# v085_rawvalue = adult uninsured (earlier releases)
# Confirm exact codes in each year's data dictionary
UNINSURED_VARIANTS = [
    'v059_rawvalue',
    'v085_rawvalue',
    'uninsured',
    '% uninsured',
    'pct_uninsured',
]

In [14]:
def detect_column(cols_lower, variants):
    """
    Return the first column name that matches any variant string.
    Tries exact match first, then partial match.
    """
    for v in variants:
        if v in cols_lower:
            return v
    for col in cols_lower:
        for v in variants:
            if v in col:
                return col
    return None

In [15]:
audit_rows = []

for _, fetch_row in fetch_df.iterrows():
    yr  = fetch_row['release_year']
    row = {'release_year': yr}

    if fetch_row['status'].startswith('ERROR') or fetch_row['path'] is None:
        row['status'] = 'FILE_MISSING'
        audit_rows.append(row)
        continue

    try:
        df_head    = pd.read_csv(fetch_row['path'], nrows=5, dtype=str, encoding='latin-1')
        cols       = list(df_head.columns)
        cols_lower = [c.strip().lower() for c in cols]

        with open(fetch_row['path'], 'r', encoding='latin-1') as f:
            n_rows = sum(1 for _ in f) - 1

        row.update({
            'status'       : 'OK',
            'n_rows'       : n_rows,
            'n_cols'       : len(cols),
            'fips_col'     : detect_column(cols_lower, FIPS_VARIANTS),
            'uninsured_col': detect_column(cols_lower, UNINSURED_VARIANTS),
            'first_3_cols' : ' | '.join(cols[:3]),
        })
    except Exception as e:
        row['status'] = f'READ_ERROR: {e}'

    audit_rows.append(row)

audit = pd.DataFrame(audit_rows)

In [16]:
audit

,release_year,status,n_rows,n_cols,fips_col,uninsured_col,first_3_cols
0,2010,OK,3194,198,5-digit fips code,uninsured adults raw value,State FIPS Code | County FIPS Code | 5-digit F...
1,2011,OK,3194,379,5-digit fips code,uninsured adults raw value,State FIPS Code | County FIPS Code | 5-digit F...
2,2012,OK,3194,330,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
3,2013,OK,3194,466,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
4,2014,OK,3194,487,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
5,2015,OK,3194,462,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
6,2016,OK,3194,482,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
7,2017,OK,3196,492,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
8,2018,OK,3195,508,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...
9,2019,OK,3195,534,5-digit fips code,uninsured raw value,State FIPS Code | County FIPS Code | 5-digit F...


In [17]:
print('FIPS column names seen across releases:')
print(audit['fips_col'].value_counts(dropna=False))

FIPS column names seen across releases:
fips_col
5-digit fips code    16
Name: count, dtype: int64


In [18]:
print('Uninsured column names seen across releases:')
print(audit['uninsured_col'].value_counts(dropna=False))

Uninsured column names seen across releases:
uninsured_col
uninsured raw value           14
uninsured adults raw value     2
Name: count, dtype: int64


In [19]:
audit.to_csv(OUTPUT_DIR / 'file_audit.csv', index=False)
print('Saved: outputs/file_audit.csv')

Saved: outputs/file_audit.csv


### Summary
Audit complete. Review the FIPS and uninsured column name counts above. Any variation across years will need explicit handling in the Week 3 column standardization step. All files with status OK are ready for panel construction.

---
## Section 4: Release Year vs. Measurement Year Reference Table

### Description
CHR release year is not the same as the survey or measurement year. The uninsured measure draws from the American Community Survey (ACS) and typically reflects surveys conducted approximately two years before the release. This table maps each release year to its approximate ACS measurement period.

The `verified` column should be updated to `True` as you confirm each entry against the corresponding technical document.

In [20]:
# Approximate ACS measurement periods per CHR release year
# Verify each entry against the release-year technical document
release_measure_map = {
    2010: '2006-2008 ACS',
    2011: '2007-2009 ACS',
    2012: '2008-2010 ACS',
    2013: '2009-2011 ACS',
    2014: '2010-2012 ACS',
    2015: '2011-2013 ACS',
    2016: '2012-2014 ACS',
    2017: '2013-2015 ACS',
    2018: '2014-2016 ACS',
    2019: '2015-2017 ACS',
    2020: '2016-2018 ACS',
    2021: '2017-2019 ACS',
    2022: '2018-2020 ACS',
    2023: '2019-2021 ACS',
    2024: '2020-2022 ACS',
    # Verify 2025 entry in the 2025 technical document
    2025: '2021-2023 ACS',
}

In [21]:
ref_table = pd.DataFrame([
    {'release_year': yr, 'approx_measure_period': v, 'verified': False}
    for yr, v in release_measure_map.items()
])

ref_table

,release_year,approx_measure_period,verified
0,2010,2006-2008 ACS,False
1,2011,2007-2009 ACS,False
2,2012,2008-2010 ACS,False
3,2013,2009-2011 ACS,False
4,2014,2010-2012 ACS,False
5,2015,2011-2013 ACS,False
6,2016,2012-2014 ACS,False
7,2017,2013-2015 ACS,False
8,2018,2014-2016 ACS,False
9,2019,2015-2017 ACS,False


In [22]:
ref_table.to_csv(OUTPUT_DIR / 'release_measure_year_map.csv', index=False)
print('Saved: outputs/release_measure_year_map.csv')

Saved: outputs/release_measure_year_map.csv


### Summary
Reference table saved. Update the `verified` column to `True` as you confirm each row against the technical documents. The measurement year column will be joined onto the panel during Week 3 so that the time axis reflects survey periods rather than release years.

---
## Section 5: Known Comparability Flags

### Description
Document known structural breaks and decisions required before panel construction. Each flag has an `action_taken` field to be filled in during Week 3 as decisions are made.

In [23]:
comparability_flags = [
    {
        'flag_id'     : 'CT_2022',
        'release_year': 2022,
        'scope'       : 'Connecticut',
        'issue'       : 'Connecticut replaced its 8 counties with 9 planning regions in 2022. '
                        'CT rows in 2022 and later do not correspond to CT rows in 2010-2021. '
                        'Decision required: exclude CT entirely, or cross-walk and flag.',
        # Fill in after reading CHR 2022 Technical Document, section on geographic changes
        'action_taken': None,
        'reference'   : 'CHR 2022 Technical Document, Geographic Changes section',
    },
    {
        'flag_id'     : 'UNINSURED_COL_RENAME',
        'release_year': None,
        'scope'       : 'All states',
        'issue'       : 'The uninsured measure column was numbered differently across releases. '
                        'Confirm the correct column code in each year data dictionary before stacking.',
        # Fill in after completing the data dictionary review in Week 3
        'action_taken': None,
        'reference'   : 'Annual CHR Data Dictionary (each release year)',
    },
    {
        'flag_id'     : 'MEASURE_YEAR_LAG',
        'release_year': None,
        'scope'       : 'All measures',
        'issue'       : 'CHR release year does not equal the survey measurement year. '
                        'Analysis must use measurement year on the time axis, not release year.',
        'action_taken': 'Scaffold built in Section 5. Verification in progress.',
        'reference'   : 'outputs/release_measure_year_map.csv',
    },
    {
        'flag_id'     : 'STATE_SUMMARY_ROWS',
        'release_year': None,
        'scope'       : 'All states',
        'issue'       : 'CHR files include one state-level summary row per state. '
                        'These rows have a FIPS county code of 000 and must be removed before panel construction.',
        'action_taken': None,
        'reference'   : 'CHR Data Dictionary, FIPS code structure',
    },
]

In [24]:
flags_df = pd.DataFrame(comparability_flags)
flags_df

,flag_id,release_year,scope,issue,action_taken,reference
0,CT_2022,2022.000,Connecticut,Connecticut replaced its 8 counties with 9 pla...,None,"CHR 2022 Technical Document, Geographic Change..."
1,UNINSURED_COL_RENAME,NaN,All states,The uninsured measure column was numbered diff...,None,Annual CHR Data Dictionary (each release year)
2,MEASURE_YEAR_LAG,NaN,All measures,CHR release year does not equal the survey mea...,Scaffold built in Section 5. Verification in p...,outputs/release_measure_year_map.csv
3,STATE_SUMMARY_ROWS,NaN,All states,CHR files include one state-level summary row ...,None,"CHR Data Dictionary, FIPS code structure"


In [25]:
flags_df.to_csv(OUTPUT_DIR / 'comparability_flags.csv', index=False)
print('Saved: outputs/comparability_flags.csv')

Saved: outputs/comparability_flags.csv


### Summary
Four comparability flags documented. The CT_2022 geographic break and the uninsured column rename are the highest-priority items to resolve in Week 3. The state summary row filter is a straightforward removal step during panel construction.

---
## Section 6: Data Source Notes

### Description
Generate a written record of the data source for inclusion in the Part 1 technical memo

In [32]:
source_notes = f"""\
DATA SOURCE NOTES
CHR Panel Study, Part 1, Week 2
Generated: {date.today().isoformat()}
==========================================================

PRIMARY DATA SOURCE
Name      : County Health Rankings and Roadmaps (CHR)
Producer  : University of Wisconsin Population Health Institute,
            funded by Robert Wood Johnson Foundation
URL       : https://www.countyhealthrankings.org
Files used: Annual CSV Analytic Data, 2010-2025 (16 releases)
License   : Publicly available, free download

FILE FORMAT
Each annual release is a single CSV file. Rows include one record per
county (identified by a 5-digit FIPS code) plus one state-summary row
per state (FIPS county portion = 000). State-summary rows will be
removed during panel construction.

UNIT OF ANALYSIS
County-year. The panel will have one row per (county FIPS, release year)
pair after stacking all sixteen files.

PANEL FILE
The consolidated panel will be saved as data/chr_panel.csv and used
as the primary data source for Parts 1, 2, and 3. Raw files remain
unmodified. Rerunning the Week 3 preprocessing notebook regenerates
the panel from scratch.

OUTCOME VARIABLE (PRELIMINARY)
Primary candidate   : % Uninsured (all ages), raw value column.
                      Column code varies by year. v059_rawvalue is used
                      in recent releases. Confirm in each year data dictionary.
Secondary candidate : % Uninsured Adults, reported separately in some years.
                      Will be evaluated in the independent extension.

MEASUREMENT LAG
CHR release year does not equal survey measurement year. The uninsured
measure draws from the ACS and typically reflects surveys conducted
approximately two years before the release year. See
outputs/release_measure_year_map.csv for the year-by-year mapping.

KNOWN STRUCTURAL BREAKS
1. Connecticut 2022: CT replaced 8 counties with 9 planning regions.
   Rows are not longitudinally comparable before and after this change.
2. Variable renaming: Uninsured measure column names changed across releases.
3. State-summary rows must be filtered out before modeling.

DOCUMENTATION TO RETAIN
For each release year, collect and store:
  - Data dictionary (variable definitions, codes, denominators)
  - Technical document (methodology, source surveys, comparability notes)
  - Any measure-change or comparability notes

DOWNLOAD DATE
2026-05-26.

ANALYST NOTES
Add notes here as you work through the files.
"""

print(source_notes)

DATA SOURCE NOTES
CHR Panel Study, Part 1, Week 2
Generated: 2026-05-26

PRIMARY DATA SOURCE
Name      : County Health Rankings and Roadmaps (CHR)
Producer  : University of Wisconsin Population Health Institute,
            funded by Robert Wood Johnson Foundation
URL       : https://www.countyhealthrankings.org
Files used: Annual CSV Analytic Data, 2010-2025 (16 releases)
License   : Publicly available, free download

FILE FORMAT
Each annual release is a single CSV file. Rows include one record per
county (identified by a 5-digit FIPS code) plus one state-summary row
per state (FIPS county portion = 000). State-summary rows will be
removed during panel construction.

UNIT OF ANALYSIS
County-year. The panel will have one row per (county FIPS, release year)
pair after stacking all sixteen files.

PANEL FILE
The consolidated panel will be saved as data/chr_panel.csv and used
as the primary data source for Parts 1, 2, and 3. Raw files remain
unmodified. Rerunning the Week 3 preprocessing 

In [27]:
with open(OUTPUT_DIR / 'data_source_notes.txt', 'w') as f:
    f.write(source_notes)

print('Saved: outputs/data_source_notes.txt')

Saved: outputs/data_source_notes.txt


### Summary
Source notes written to file. Update the download date and analyst notes fields as you progress. This document will be referenced in the Part 1 technical memo.

---
## Section 7: Initial Research Questions

### Description
Formalize the four research questions from the Part 1 guideline. Each question is linked to relevant variables, the planned method, and the week it will be addressed.

In [28]:
research_questions = [
    {
        'rq_id'               : 'RQ1',
        'question'            : 'Which CHR uninsured measure is most appropriate as the main outcome '
                                'for a longitudinal county-level analysis?',
        'relevant_variables'  : 'v059_rawvalue (% uninsured all ages); v085_rawvalue (% uninsured adults)',
        'method'              : 'Data dictionary review; availability table; measurement consistency check',
        'week'                : '2-3',
    },
    {
        'rq_id'               : 'RQ2',
        'question'            : 'Which socioeconomic and demographic variables are available consistently '
                                'enough to support analysis across the full 2010-2025 panel?',
        'relevant_variables'  : 'Poverty rate, median household income, unemployment, '
                                'educational attainment, race/ethnicity composition, rurality, '
                                'Medicaid expansion status (external join)',
        'method'              : 'Variable availability table; missingness summary',
        'week'                : '2-3',
    },
    {
        'rq_id'               : 'RQ3',
        'question'            : 'How did uninsured rates change nationally, across states, and across '
                                'counties from 2010 through 2025?',
        'relevant_variables'  : 'Uninsured rate (outcome); year; state FIPS; county FIPS',
        'method'              : 'Descriptive trend analysis; national and state-level plots; '
                                'distribution comparisons across time',
        'week'                : '4',
    },
    {
        'rq_id'               : 'RQ4',
        'question'            : 'What do basic regression and prediction models reveal about the '
                                'structure of the uninsured-rate problem, and what do they leave unanswered?',
        'relevant_variables'  : 'All panel variables after missingness filtering',
        'method'              : 'OLS; ridge regression; lasso; random forest',
        'week'                : '5',
    },
]

In [29]:
rq_df = pd.DataFrame(research_questions)
rq_df

,rq_id,question,relevant_variables,method,week
0,RQ1,Which CHR uninsured measure is most appropriat...,v059_rawvalue (% uninsured all ages); v085_raw...,Data dictionary review; availability table; me...,2-3
1,RQ2,Which socioeconomic and demographic variables ...,"Poverty rate, median household income, unemplo...",Variable availability table; missingness summary,2-3
2,RQ3,"How did uninsured rates change nationally, acr...",Uninsured rate (outcome); year; state FIPS; co...,Descriptive trend analysis; national and state...,4
3,RQ4,What do basic regression and prediction models...,All panel variables after missingness filtering,OLS; ridge regression; lasso; random forest,5


In [30]:
rq_df.to_csv(OUTPUT_DIR / 'research_questions.csv', index=False)
print('Saved: outputs/research_questions.csv')

Saved: outputs/research_questions.csv


### Summary
Four research questions documented and saved. RQ1 and RQ2 drive Week 3 panel construction decisions. RQ3 drives Week 4 descriptive analysis. RQ4 drives Week 5 baseline modeling.

---
## Section 8: Notebook Summary

### Description
Summary of all work completed in this notebook and outputs produced.

### Summary

**What was done in Week 2:**

| Section | Work | Output |
|---|---|---|
| 2 | Downloaded all 16 CHR CSV files from GitHub and cached locally | `data/raw/` |
| 3 | Audited column names and row counts per file | `outputs/file_audit.csv` |
| 4 | Built release year to measurement year scaffold | `outputs/release_measure_year_map.csv` |
| 5 | Documented four comparability flags | `outputs/comparability_flags.csv` |
| 6 | Wrote data source notes for the Part 1 memo | `outputs/data_source_notes.txt` |
| 7 | Formalized four research questions | `outputs/research_questions.csv` |

**Key decisions made:**
- The consolidated panel will be saved as `data/chr_panel.csv` and used as the single data source for Parts 1, 2, and 3.
- FIPS codes will be treated as strings throughout.
- State-summary rows (county FIPS = 000) will be removed during panel construction.
- The measurement year column will be derived from the reference table and used as the time axis.

**Outstanding items before Week 3:**
- Download any missing CHR CSV files.
- Download data dictionaries and technical documents for all 16 release years.
- Verify measurement year entries in `release_measure_year_map.csv`.
- Confirm correct uninsured column code for each release year and update `UNINSURED_VARIANTS`.
- Read the Connecticut 2022 geographic change note and decide on exclusion vs. crosswalk.

**Next notebook:** Part 1: Week 3 -  Panel Construction and Data Audit.